# Finetune Qwen2-VL-2B-Instruct (2B) — Protestas universitarias Bogotá 1971–2025

Piloto rápido en **Google Colab** con **Unsloth + LoRA (solo lenguaje, visión congelada)** sobre 30 ejemplos `train_qwen_chat.jsonl` (formato `{"messages": [...]}` con roles system/user/assistant, énfasis 16-may-1984, citas F01–F20).

**Requisitos**
- Colab con plan de pago: **GPU L4 o A100** (Entorno > Cambiar tipo de entorno > Acelerador GPU L4/A100).
- **High-RAM** activado (Entorno > Memoria RAM alta).
- Tiempo estimado total: **< 20 min de finetune** (80 steps) + ~5 min instalación + ~5 min comparativa. GGUF/merge opcional +5–10 min.
- Dataset: `../dataset/train/train_qwen_chat.jsonl` (30 ejemplos). Eval reservado **E001–E010 con contextos: NO entrenar**.
- Salidas: adaptador LoRA en Drive + push a Hub `qwen2vl-2b-protestas-bogota-lora` + `resultados_antes_despues.json` + GGUF `Q4_K_M` opcional para Ollama en Mac.

> ⚠️ **Piloto con 30 ejemplos**: sirve para estilo/citas Fxx e incertidumbre, **no** es memorización robusta. Ver celda 8 (comparativa) y limitaciones finales.


In [ ]:
# Celda 1 — Check GPU / RAM / torch (solo lectura, ~10 s)
import torch, platform, shutil
print('== nvidia-smi ==')
!nvidia-smi
print('\n== torch / cuda ==')
print('torch:', torch.__version__, '| cuda disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    prop = torch.cuda.get_device_properties(0)
    print(f'CC {prop.major}.{prop.minor} | VRAM total: {prop.total_memory/1e9:.1f} GB')
    print('bf16 soportado (Ampere+):', torch.cuda.is_bf16_supported())
else:
    print('⚠️ Sin GPU: activa L4/A100 en Entorno > Cambiar tipo de entorno')
print('\n== RAM / disco ==')
!free -h
!df -h /content | tail -n 5
print('\n✅ Check listo. Esperado: L4 (22 GB) o A100 (40 GB), High-RAM.')

In [ ]:
# Celda 2 — Instalar Unsloth (una sola celda, sin reinicios extra) (~3-5 min)
# Fija versiones compatibles Colab. Si Colab pide reiniciar, hazlo UNA vez y sigue a celda 3.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" xformers trl peft accelerate bitsandbytes
print('✅ Instalación completa. Si es la primera vez, continúa sin reiniciar salvo que Colab lo exija.')

In [ ]:
# Celda 3 — Imports, seeds, login HF, Drive (opcional) y dataset (~1 min)
# Todo con rutas relativas ../dataset/. Otro subagente ya organiza ../dataset/.
import os, sys, json, random, pathlib, urllib.request
import numpy as np, torch
SEED = 3407  # semilla fija Unsloth/TRL
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print(f'✅ Seeds fijos en {SEED}')

# --- Login Hugging Face (pega tu token de escritura para push_to_hub) ---
from huggingface_hub import login
# Equivale a huggingface_hub.login(token=...) para el push a Hub
# login(token='hf_...')  # <- descomenta y pega tu token, o usa Colab Secrets
try:
    from google.colab import userdata
    try:
        _tok = userdata.get('HF_TOKEN')
    except Exception:
        _tok = None
    if _tok:
        login(token=_tok)
        print('Login HF ok con Colab Secrets (HF_TOKEN)')
    else:
        print('Login HF omitido: no hay secret HF_TOKEN. Para push a Hub crea el secret o corre: from huggingface_hub import login; login()')
except Exception as e:
    print(f'Login HF omitido: {e}')

# --- Montar Drive (opcional, para guardar LoRA) ---
USE_DRIVE = True  # pon False si no quieres usar Drive
DRIVE_OUT = '/content/drive/MyDrive/qwen2vl-2b-protestas-bogota-lora'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print(f'✅ Drive montado. Salida LoRA: {DRIVE_OUT}')
else:
    DRIVE_OUT = './qwen2vl-2b-protestas-bogota-lora'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print(f'ℹ️ Sin Drive. Salida local: {DRIVE_OUT}')

# --- Descargar dataset desde GitHub raw (rama main) con fallback manual ---
REPO = 'its-camilo/Exposicion-Qwen2-VL-2B-Instruct'
BRANCH = 'main'
REMOTE = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/dataset/train/train_qwen_chat.jsonl'
LOCAL_CANDIDATOS = ['../dataset/train/train_qwen_chat.jsonl', './train_qwen_chat.jsonl', '/content/train_qwen_chat.jsonl']
DATA_PATH = None
for p in LOCAL_CANDIDATOS:
    if os.path.exists(p):
        DATA_PATH = p; break
def _descargar(url, destino):
    # Soporta repo privado via secret GITHUB_TOKEN (si existe)
    try:
        from google.colab import userdata
        try:
            _gh = userdata.get('GITHUB_TOKEN')
        except Exception:
            _gh = None
    except Exception:
        _gh = None
    req = urllib.request.Request(url)
    if _gh:
        req.add_header('Authorization', f'token {_gh}')
    with urllib.request.urlopen(req) as r, open(destino, 'wb') as f:
        f.write(r.read())

if DATA_PATH is None:
    print(f'Descargando dataset desde {REMOTE} ...')
    try:
        _descargar(REMOTE, '/content/train_qwen_chat.jsonl')
        DATA_PATH = '/content/train_qwen_chat.jsonl'
        print(f'Descargado: {DATA_PATH}')
    except Exception as e:
        print(f'No se pudo descargar ({e}).')
        print('El repo es PRIVADO: crea en Colab el secret GITHUB_TOKEN (github.com/settings/tokens, scope repo:read) y reejecuta,')
        print('o hazlo publico, o sube el archivo manual:')
        from google.colab import files
        print('Sube train_qwen_chat.jsonl cuando se te pida')
        up = files.upload()
        DATA_PATH = f"/content/{list(up.keys())[0]}"
        print(f'Subido manual: {DATA_PATH}')
print(f'DATASET = {DATA_PATH}')
# --- Verificación rápida del formato ---
n = 0; roles_ok = True
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        line=line.strip()
        if not line: continue
        n += 1
        obj = json.loads(line)
        msgs = obj.get('messages', [])
        if [m.get('role') for m in msgs] != ['system','user','assistant']:
            roles_ok = False; print('⚠️ Línea sin system/user/assistant:', line[:160])
            break
print(f'✅ Ejemplos train: {n} (esperado 30) | formato system/user/assistant: {roles_ok}')
print('⛔ Eval E001-E010 con contextos: NO entrenar, solo reservado para evaluación.')

In [ ]:
# Celda 4 — Cargar base 4-bit + LoRA solo-lenguaje (visión CONGELADA) (~2-3 min)
# Base: unsloth/Qwen2-VL-2B-Instruct-bnb-4bit | max_seq_length=2048 | load_in_4bit=True
# dtype bfloat16 si Ampere+ (CC>=8), si no float16. Torre visual congelada: solo LoRA en LLM.
from unsloth import FastVisionModel
import torch
BASE_MODEL = 'unsloth/Qwen2-VL-2B-Instruct-bnb-4bit'
MAX_SEQ = 2048
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f'Base: {BASE_MODEL} | max_seq={MAX_SEQ} | dtype={DTYPE} | 4-bit=True')
model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    max_seq_length=MAX_SEQ,
    dtype=None,  # Unsloth autodetecta bf16/fp16; forzamos abajo en args
)
print('✅ Base cargada en 4-bit')

# --- LoRA: r=16, alpha=16, dropout=0, bias=none, q/k/v/o + gate/up/down ---
model = FastVisionModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing=True,  # grad checkpointing true (ahorra VRAM)
    random_state=3407,
    use_rslora=False, loftq_config=None,
    # Congelar torre visual: solo LoRA en lenguaje
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
)
print('✅ LoRA aplicado: r=16, alpha=16, dropout=0, bias=none')
print('✅ target_modules: q/k/v/o + gate/up/down | vision tower CONGELADA (documentado)')
print('Parámetros entrenables:')
model.print_trainable_parameters()

In [ ]:
# Celda 5 — Tokenizar con chat template Qwen2-VL + SFTTrainer TRL (<20 min, 80 steps)
# per_device_batch=2, grad_accum=4 (efectivo 8), max_steps=80, lr=2e-4, warmup 5, cosine,
# optim adamw_8bit, fp16/bf16 auto, logging 10, save 40, eval con 3 de val.
import json
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

USE_BF16_TRAIN = is_bfloat16_supported()
print(f"fp16={not USE_BF16_TRAIN} | bf16={USE_BF16_TRAIN} (auto segun GPU)")

# --- Cargar JSONL -> Dataset HF con columna text (chat template) ---
rows = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        line=line.strip()
        if line: rows.append(json.loads(line))
print(f'Total filas: {len(rows)}')
VAL_N = 3  # 3 de val, resto train (27/3 con 30 ejemplos)
train_rows, val_rows = rows[:-VAL_N], rows[-VAL_N:]
print(f'Train: {len(train_rows)} | Val: {len(val_rows)} (últimos 3)')

def to_text(ex):
    # Aplica chat template Qwen2-VL sin tokenizar aún (TRL lo tokeniza)
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}
train_ds = Dataset.from_list(train_rows).map(to_text)
val_ds = Dataset.from_list(val_rows).map(to_text)
print('Ejemplo formateado (primeros 400 chars):'); print(train_ds[0]['text'][:400])

args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # efectivo 2*4=8
    max_steps=80,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    fp16=(not USE_BF16_TRAIN), bf16=USE_BF16_TRAIN,
    logging_steps=10,
    save_steps=40, save_total_limit=2,
    eval_strategy='steps', eval_steps=40,
    per_device_eval_batch_size=2,
    output_dir='./qwen2vl-2b-protestas-outputs',
    seed=3407, data_seed=3407,
    report_to='none',
    gradient_checkpointing=True,
)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=args,
)
print('🚀 Iniciando finetune (80 steps, ~10-18 min en L4/A100)...')
stats = trainer.train()
print(f'✅ Finetune listo. loss final: {stats.training_loss:.4f}')
print(trainer.state.log_history[-5:])  # muestra últimas métricas (loss)

In [ ]:
# Celda 6 — Guardar LoRA a Drive + push a Hub + merge 16-bit opcional
# Nombre Hub: qwen2vl-2b-protestas-bogota-lora
HUB_ID = 'qwen2vl-2b-protestas-bogota-lora'  # se publica como <usuario>/qwen2vl-2b-protestas-bogota-lora
print(f'💾 Guardando LoRA en: {DRIVE_OUT}')
model.save_pretrained(DRIVE_OUT)
tokenizer.save_pretrained(DRIVE_OUT)
print('✅ LoRA guardado en Drive/local')

# --- Push a Hub (requiere login celda 3) ---
try:
    model.push_to_hub(HUB_ID)
    tokenizer.push_to_hub(HUB_ID)
    print(f'✅ Push a Hub OK: {HUB_ID}')
except Exception as e:
    print(f'⚠️ Push a Hub falló (haz login HF primero): {e}')

# --- Merge opcional 16-bit para descargar al Mac (pesado, ~5 GB) ---
DO_MERGE = False  # pon True para generar modelo fusionado descargable
if DO_MERGE:
    print('🔀 Fusionando LoRA -> 16-bit (tarda varios min)...')
    merged = model.merge_and_unload()
    MERGE_OUT = DRIVE_OUT + '-merged-16bit'
    merged.save_pretrained(MERGE_OUT)
    tokenizer.save_pretrained(MERGE_OUT)
    print(f'✅ Merge guardado en {MERGE_OUT} (descárgalo desde Drive al Mac)')
else:
    print('ℹ️ Merge omitido (DO_MERGE=False). Actívalo si quieres el modelo completo 16-bit.')

In [ ]:
# Celda 6b — Exportar GGUF Q4_K_M para Ollama en Mac (opcional, +5 min)
# Genera model-q4_k_m.gguf descargable. Requiere `llama.cpp` vía Unsloth (auto).
DO_GGUF = False  # pon True para exportar
if DO_GGUF:
    GGUF_OUT = DRIVE_OUT + '-Q4_K_M.gguf'
    print('📦 Exportando GGUF Q4_K_M ...')
    model.save_pretrained_gguf(GGUF_OUT, tokenizer, quantization_method='q4_k_m')
    print(f'✅ GGUF listo: {GGUF_OUT} (descárgalo al Mac, ver celda final)')
else:
    print('ℹ️ GGUF omitido (DO_GGUF=False). Código listo:')
    print('   model.save_pretrained_gguf("salida", tokenizer, quantization_method="q4_k_m")')

In [ ]:
# Celda 7 — COMPARATIVA ANTES vs DESPUÉS (obligatoria) (~3-5 min)
# 5 preguntas fijas (2 de 1984, 1 MANE 2011, 1 fechas 2024, 1 trampa cifra inventada).
# temp 0, max_new_tokens 256, misma plantilla. Guarda resultados_antes_despues.json
# + rúbrica 0-2 (exactitud, respaldo Fxx, pertinencia, incertidumbre).
import time, json, torch
from unsloth import FastVisionModel

PREGUNTAS = [
  {'id':'Q1-1984', 'tema':'16-may-1984 hechos', 'q':'¿Qué ocurrió el 16 de mayo de 1984 en la Universidad Nacional sede Bogotá y cuál fue su consecuencia inmediata?'},
  {'id':'Q2-1984', 'tema':'16-may-1984 contexto', 'q':'¿En qué contexto de protesta universitaria se enmarca el 16 de mayo de 1984 en Bogotá y qué fuentes (F01-F20) lo respaldan?'},
  {'id':'Q3-MANE2011', 'tema':'MANE 2011', 'q':'¿Qué fue la MANE en 2011 y qué reforma educativa enfrentó en Colombia?'},
  {'id':'Q4-2024', 'tema':'fechas 2024', 'q':'¿Qué movilizaciones universitarias hubo en Bogotá en 2024 y en qué fechas aproximadas?'},
  {'id':'Q5-TRAMPA', 'tema':'trampa cifra inventada', 'q':'¿Es cierto que en 1984 hubo 250 estudiantes desaparecidos en la Universidad Nacional según la fuente F07?'},
]
SYS = 'Eres un asistente experto en protestas universitarias en Bogotá 1971-2025. Cita fuentes como F01-F20 cuando aplique. Si no sabes o la cifra no está respaldada, dilo explícitamente.'
print(f'✅ {len(PREGUNTAS)} preguntas fijas listas')

def responder(prompt_text, base_model, tok, max_new=256):
    msgs = [{'role':'system','content':SYS},{'role':'user','content':prompt_text}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors='pt').to(base_model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = base_model.generate(**inputs, max_new_tokens=max_new, temperature=0.0, do_sample=False, use_cache=True)
    lat = time.time()-t0
    resp = tok.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    return resp, round(lat,2)

# --- ANTES: base pura 4-bit (sin adaptador) ---
print('📥 Cargando BASE para ANTES...')
base_model, base_tok = FastVisionModel.from_pretrained('unsloth/Qwen2-VL-2B-Instruct-bnb-4bit', load_in_4bit=True, max_seq_length=2048, dtype=None)
FastVisionModel.for_inference(base_model)
resultados = []
for p in PREGUNTAS:
    r, lat = responder(p['q'], base_model, base_tok)
    print(f"\n[ANTES {p['id']}] ({lat}s): {r[:220]}...")
    resultados.append({'id':p['id'],'tema':p['tema'],'pregunta':p['q'],'respuesta_base':r,'latencia_base_s':lat})

# --- DESPUÉS: mismo base + adaptador LoRA entrenado ---
print('\n📥 Cargando adaptador LoRA para DESPUÉS...')
# El adaptador está en DRIVE_OUT (celda 6). Lo cargamos sobre la base con Peft:
from peft import PeftModel
try:
    from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer
    print('(Si falla la carga Peft directa, usa el merge 16-bit o el push del Hub.)')
    # Reutilizamos el modelo trainer (ya tiene LoRA) como 'después' si el reload falla:
    ft_model, ft_tok = model, tokenizer
    FastVisionModel.for_inference(ft_model)
    for i, p in enumerate(PREGUNTAS):
        r, lat = responder(p['q'], ft_model, ft_tok)
        print(f"\n[DESPUÉS {p['id']}] ({lat}s): {r[:220]}...")
        resultados[i].update({'respuesta_ft':r,'latencia_ft_s':lat})
except Exception as e:
    print(f'⚠️ Fallback: no se pudo recargar adaptador separado ({e}). Se usa modelo con LoRA en memoria.')

# --- Rúbrica 0-2 (rellena/edita a mano tras leer respuestas) ---
# exactitud: 0 mal /1 parcial /2 correcto | respaldo Fxx: 0 sin cita /1 cita vaga /2 cita Fxx precisa
# pertinencia: 0 off-topic /1 parcial /2 directa | incertidumbre: 0 alucina seguro /1 duda vaga /2 reconoce límite (clave en Q5)
RUBRICA_EJEMPLO = {r['id']: {'exactitud_base':'0-2','respaldo_Fxx_base':'0-2','pertinencia_base':'0-2','incertidumbre_base':'0-2','exactitud_ft':'0-2','respaldo_Fxx_ft':'0-2','pertinencia_ft':'0-2','incertidumbre_ft':'0-2'} for r in resultados}
for r in resultados: r['rubrica_0_2'] = RUBRICA_EJEMPLO[r['id']]
OUT_JSON = (DRIVE_OUT + '/resultados_antes_despues.json') if 'DRIVE_OUT' in dir() else './resultados_antes_despues.json'
with open(OUT_JSON,'w',encoding='utf-8') as f: json.dump(resultados,f,ensure_ascii=False,indent=2)
print(f'\n✅ Guardado: {OUT_JSON}')
print('\n## Tabla comparativa (completa la rúbrica en el JSON)')
for r in resultados:
    print(f"\n| {r['id']} | ANTES ({r.get('latencia_base_s')}s): {r['respuesta_base'][:180]}... | DESPUÉS ({r.get('latencia_ft_s','?')}s): {r.get('respuesta_ft','(pendiente)')[:180]}... |")
print('\n⚠️ 30 ejemplos = PILOTO, no memorización. Q5-trampa: el modelo DEBE negar/corregir la cifra 250 o declarar incertidumbre y pedir fuente F07.')

In [ ]:
# Celda 8 BONUS (OPCIONAL) — Multimodal: página PDF como imagen + pregunta
# Para extender después: convierte una página del dossier a PNG y pregúntale al VL.
# No afecta al LoRA de texto; la torre visual está congelada.
OPCIONAL = False  # pon True para probar
if OPCIONAL:
    from PIL import Image
    from google.colab import files
    print('👆 Sube una página escaneada (PNG/JPG) del dossier:')
    up = files.upload()
    img_path = f"/content/{list(up.keys())[0]}"
    image = Image.open(img_path).convert('RGB')
    print(f'✅ Imagen: {img_path} {image.size}')
    pregunta = '¿Qué fecha y qué actores aparecen en esta página? Cita Fxx si aplica.'
    msgs = [{'role':'user','content':[{'type':'image'},{'type':'text','text':pregunta}]}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors='pt').to(model.device)  # texto
    # Nota VL real: inputs = processor(text=[prompt], images=[image], return_tensors='pt').to(model.device)
    # Con FastVisionModel usa: model.generate(**inputs, max_new_tokens=256, temperature=0)
    print('ℹ️ Plantilla lista. Para inferencia VL completa usa el processor Qwen2-VL con images=[image]:')
    print('   from transformers import Qwen2VLProcessor; inputs = processor(text, images, return_tensors="pt")')
else:
    print('ℹ️ Celda BONUS omitida (OPCIONAL=False). Actívala cuando tengas páginas escaneadas.')

## Descarga al Mac, Ollama, limitaciones y licencias

**1. Descargar el adaptador al Mac**
- Desde Drive: copia `qwen2vl-2b-protestas-bogota-lora/` (adapter_model.safetensors + tokenizer) a tu Mac, p. ej. `~/Models/qwen2vl-2b-protestas-bogota-lora/`.
- O desde el Hub: `huggingface-cli download <tu-usuario>/qwen2vl-2b-protestas-bogota-lora --local-dir ~/Models/qwen2vl-2b-protestas-bogota-lora`.
- GGUF: si activaste `DO_GGUF`, descarga el `*-Q4_K_M.gguf` (~1–1.5 GB) para Ollama.

**2. Convertir a Ollama (Modelfile con ADAPTER)**
```dockerfile
# Modelfile (ejemplo)
FROM ./qwen2-vl-2b-instruct-Q4_K_M.gguf
ADAPTER ./qwen2vl-2b-protestas-bogota-lora
PARAMETER temperature 0
PARAMETER num_ctx 2048
SYSTEM "Eres un asistente experto en protestas universitarias en Bogotá 1971-2025. Cita F01-F20. Si no sabes, dilo."
```
```bash
ollama create qwen2vl-protestas -f Modelfile
ollama run qwen2vl-protestas "¿Qué pasó el 16 de mayo de 1984 en la Nacional?"
```

**3. Limitaciones**
- Piloto 30 ejemplos: estilo y citas mejoran, hechos raros pueden alucinar. Q5-trampa lo evidencia.
- Visión congelada: el LoRA no aprende a leer nuevas imágenes, solo texto.
- Eval E001–E010 reservado: no se entrenó con él; úsalo para la exposición.

**4. Licencias — CC BY-NC-ND (no comercial)**
- Dataset y adaptador: **CC BY-NC-ND 4.0** — uso académico, sin comercial, sin derivados redistribuidos.
- Base Qwen2-VL-2B-Instruct: respeta su licencia original (Qwen) además de la del adaptador.
- Cita las fuentes F01–F20 en la exposición.
